In [1]:
import torch
import torch.nn as nn

from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)

DEVICE = torch.device("cpu")

model = efficientnet_b0(
    weights=None
)

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    2
)

model.load_state_dict(
    torch.load(
        "document_classifier.pth",
        map_location=DEVICE
    )
)

model.eval()

print("Model loaded")

Model loaded


In [2]:
class_names = [
    "aadhaar",
    "pan"
]
from PIL import Image
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

def predict_document(image_path):

    image = Image.open(
        image_path
    ).convert("RGB")

    image = transform(
        image
    ).unsqueeze(0)

    with torch.no_grad():

        output = model(
            image
        )

        probs = torch.softmax(
            output,
            dim=1
        )

        confidence, pred = torch.max(
            probs,
            1
        )

    return (
        class_names[
            pred.item()
        ],
        confidence.item()
    )

In [4]:
doc_type, conf = predict_document(
    "C:\\Users\\nitis\\Downloads\\Nitish_aadhar.png"
)

print(
    "Prediction:",
    doc_type
)

print(
    "Confidence:",
    round(conf * 100, 2),
    "%"
)

Prediction: aadhaar
Confidence: 99.54 %
